NOTES
- some .mat files are just 1 KB, whereas majority should be around 25 000 KB
- 7, 10, 26 and 49 are the only ones with this issue, and they seem to be the ones where each actioni is a (0,0) ndarray
- check if they are stored by another version of matlab or just plain empty of data. not horrible if they are, as only 4 subjects. 
- 6 additional clips/actions could not be read, from e.g., subject 8. 
- No clips were skipped due to too short clip. 
- need to identify time stamp tags for synchronization... 
- work on reading the h5 file to verify the total nbr of minutes loaded, is 4 missing participants a big issue?


In [1]:
import random
from pathlib import Path

import h5py
import numpy as np
import scipy.io as sio

p1 = "F_AMASS/F_amass_Subject_1.mat"
p10 = "F_AMASS/F_amass_Subject_10.mat"

m1 = sio.loadmat(str(p1), struct_as_record=False, squeeze_me=False)



In [14]:
def _unwrap(x):
    """
    Unwrap nested numpy object arrays from scipy.io.loadmat (squeeze_me=False).
    Only peels single-element (size==1) object arrays, so multi-element arrays
    like move (21,1) are left intact. Stops when we reach a mat_struct,
    a numeric ndarray, or a scalar -- regardless of how many layers deep.
    """
    while isinstance(x, np.ndarray) and x.dtype == object and x.size == 1:
        x = x.flat[0]
    return x


def _scalar(x) -> int | float:
    """Extract a Python scalar from any numpy array shape."""
    if isinstance(x, np.ndarray):
        return x.flat[0].item()
    return x


def _str(x) -> str:
    """Extract a plain Python str from a numpy string scalar or 1-element array."""
    if isinstance(x, np.ndarray):
        return str(x.flat[0])
    return str(x)


def load_amass_mat(path: Path) -> tuple[dict, list[dict]] | tuple[None, None]:
    """
    Load one F_amass_Subject_X.mat.

    Uses squeeze_me=False and explicit unwrapping to be robust across
    all scipy versions and all 90 subject files.

    Returns
    -------
    meta  : dict        — subject metadata
    clips : list[dict]  — one dict per action clip
    """
    try:
        mat = sio.loadmat(str(path), struct_as_record=False, squeeze_me=False)
    except Exception as e:
        # log.warning(f"Cannot load {path.name}: {e}")
        return None, None

    top_key = next(k for k in mat if not k.startswith("__"))
    subj    = _unwrap(mat[top_key])   # mat_struct with fields: id, subject, move
    s       = _unwrap(subj.subject)   # mat_struct with fields: id, sex, height, ...

    meta = {
        "id":     _str(s.id),
        "gender": _str(s.sex),
        "height": int(_scalar(s.height)),
        "mass":   int(_scalar(s.mass)),
        "age":    int(_scalar(s.age)),
    }

    # move_arr shape varies across subjects: (21,1), (1,21), etc.
    # Iterate .flat so we never assume a particular axis layout.
    move_arr = subj.move

    # print(top_key)

    # print(subj)
    # print(s)
    # print(type(move_arr))
    # print(type(move_arr.flat[0]))
    # print(move_arr.shape)

    # print(type(move_arr.flat[0].flat[0]))

    # return None, None
    clips = []
    cell_counter = 1
    for cell in move_arr.flat:
        action = _unwrap(cell)

        # Some subjects have extra nesting layers — keep peeling
        # until we reach a mat_struct or give up
        max_depth = 10
        depth = 0
        while not hasattr(action, "_fieldnames") and depth < max_depth:
            if isinstance(action, np.ndarray) and action.size > 0:
                action = action.flat[0]
            else:
                print(type(action))
                # print(action._fieldnames)
                print(action.shape)
                # print(action.flat[0])
                return meta, action
                # raise Exception
                # break
            depth += 1

        if not hasattr(action, "_fieldnames"):
            print(f"  Could not unwrap action cell in {meta['id']}, skipping. "
                         f"Final type: {type(action).__name__}")
            print(f'Cell counter {cell_counter}')
            print(f'depth: {depth}')
            print()
            print()
            # log.warning(f"  Could not unwrap action cell in {meta['id']}, skipping. "
            #             f"Final type: {type(action).__name__}")
            continue
        else:
            # print(action.description)
            # print(action.__dict__)
            print(type(action))
            print(action._fieldnames)
            print()
            # print(f"Unwrapped cell in {meta['id']},  "
            #              f"Final type: {type(action)}")
            # print(f'Cell counter {cell_counter}')
            # print(f'depth: {depth}')
            # print()
            # print()
            # print(action.description)

        poses = action.jointsExpMaps_amass.astype(np.float32)    # (T, 52, 3)
        trans = action.RootTranslation_amass.astype(np.float32)  # (T, 3)
        betas = action.jointsBetas_amass.astype(np.float32).reshape(16)  # (16,)

        clips.append({
            "action": _str(action.description),
            "poses":  poses,
            "trans":  trans,
            "betas":  betas,
            "T":      poses.shape[0],
        })

    return meta, clips

In [15]:

meta1, clips1 = load_amass_mat(p1)
tot_frames = 0

# meta10, clips10 = load_amass_mat(p10)
# for clip in clips1:
#     T = clip["T"]
#     print(f'{clip["action"]}: {T} frames, {T/120} seconds, {T/120/60} minutes')
#     tot_frames += T
# secs = tot_frames/120
# whole_mins = int(secs/60)

# print(f'{tot_frames} frames, {tot_frames/120} seconds, {whole_mins} whole mins')

<class 'scipy.io.matlab._mio5_params.mat_struct'>
['RootTranslation_amass', 'jointsBetas_amass', 'jointsLocation_amass', 'jointsExpMaps_amass', 'jointsParent', 'description']

<class 'scipy.io.matlab._mio5_params.mat_struct'>
['RootTranslation_amass', 'jointsBetas_amass', 'jointsLocation_amass', 'jointsExpMaps_amass', 'jointsParent', 'description']

<class 'scipy.io.matlab._mio5_params.mat_struct'>
['RootTranslation_amass', 'jointsBetas_amass', 'jointsLocation_amass', 'jointsExpMaps_amass', 'jointsParent', 'description']

<class 'scipy.io.matlab._mio5_params.mat_struct'>
['RootTranslation_amass', 'jointsBetas_amass', 'jointsLocation_amass', 'jointsExpMaps_amass', 'jointsParent', 'description']

<class 'scipy.io.matlab._mio5_params.mat_struct'>
['RootTranslation_amass', 'jointsBetas_amass', 'jointsLocation_amass', 'jointsExpMaps_amass', 'jointsParent', 'description']

<class 'scipy.io.matlab._mio5_params.mat_struct'>
['RootTranslation_amass', 'jointsBetas_amass', 'jointsLocation_amass',

In [ ]:
meta1

({'id': 'Subject_1', 'gender': 'male', 'height': 184, 'mass': 92, 'age': 25},
 {'id': 'Subject_10',
  'gender': 'female',
  'height': 175,
  'mass': 63,
  'age': 24})

In [22]:
path = "F_Subjects_1_45/F_v3d_Subject_1.mat"

mat = sio.loadmat(str(path), struct_as_record=False, squeeze_me=False)
top_key = next(k for k in mat if not k.startswith("__"))
subj    = _unwrap(mat[top_key])   # mat_struct with fields: id, subject, move
s       = _unwrap(subj.subject)   # mat_struct with fields: id, sex, height, ...

meta = {
    "id":     _str(s.id),
    "gender": _str(s.sex),
    "height": int(_scalar(s.height)),
    "mass":   int(_scalar(s.mass)),
    "age":    int(_scalar(s.age)),
}
mat
meta
subj.__dict__
subj.move
# mat[top_key].move
# subj.__dict__
# s.__dict__
# meta, clips = load_amass_mat(path)
move_arr = _unwrap(subj.move)
move_arr.__dict__


{'_fieldnames': ['description',
  'markerName',
  'markerType',
  'markerDescription',
  'markerSide',
  'markerMatch',
  'segmentName',
  'segmentType',
  'segmentDescription',
  'segmentSide',
  'segmentMatch',
  'segmentParent',
  'physicalMarkers',
  'virtualMarkers',
  'lcData',
  'flags30',
  'flags120',
  'motions_list',
  'markerGaps',
  'virtualMarkerParent',
  'markerLocation',
  'virtualMarkerLocation',
  'jointsAffine_v3d',
  'jointsTranslation_v3d',
  'jointsExpMaps_v3d',
  'jointsGaps_v3d'],
 'description': array(['All_motions_V3D'], dtype='<U15'),
 'markerName': array([[array(['C7'], dtype='<U2'), array(['T10'], dtype='<U3'),
         array(['CLAV'], dtype='<U4'), array(['STRN'], dtype='<U4'),
         array(['MBLLY'], dtype='<U5'), array(['LFHD'], dtype='<U4'),
         array(['LBHD'], dtype='<U4'), array(['LBAK'], dtype='<U4'),
         array(['LSHO'], dtype='<U4'), array(['LUPA'], dtype='<U4'),
         array(['LELB'], dtype='<U4'), array(['LFRM'], dtype='<U4'),
     

In [134]:
path = p10

mat = sio.loadmat(str(path), struct_as_record=False, squeeze_me=False)
top_key = next(k for k in mat if not k.startswith("__"))
subj    = _unwrap(mat[top_key])   # mat_struct with fields: id, subject, move
s       = _unwrap(subj.subject)   # mat_struct with fields: id, sex, height, ...

meta = {
    "id":     _str(s.id),
    "gender": _str(s.sex),
    "height": int(_scalar(s.height)),
    "mass":   int(_scalar(s.mass)),
    "age":    int(_scalar(s.age)),
}
mat


{'__header__': b'MATLAB 5.0 MAT-file, Platform: GLNXA64, Created on: Thu Oct 31 11:20:12 2019',
 '__version__': '1.0',
 '__globals__': [],
 'Subject_10_F_amass': array([[<scipy.io.matlab._mio5_params.mat_struct object at 0x000001E52514D900>]],
       dtype=object)}

In [135]:
mat = sio.loadmat(str(path), struct_as_record=False, squeeze_me=True)

top_key = next(k for k in mat if not k.startswith("__"))
subj    = _unwrap(mat[top_key])   # mat_struct with fields: id, subject, move
s       = _unwrap(subj.subject)   # mat_struct with fields: id, sex, height, ...

meta = {
    "id":     _str(s.id),
    "gender": _str(s.sex),
    "height": int(_scalar(s.height)),
    "mass":   int(_scalar(s.mass)),
    "age":    int(_scalar(s.age)),
}
mat
meta
mat[top_key].move
subj.__dict__
s.__dict__

{'_fieldnames': ['id', 'sex', 'handedness', 'height', 'mass', 'age'],
 'id': 'Subject_10',
 'sex': 'female',
 'handedness': 'right',
 'height': 175,
 'mass': 63,
 'age': 24}

In [4]:
h5file = "Gmovi.h5"

with h5py.File(h5file,"r") as h5f: 
        for split in ("train", "val", "test"):
            keys = list(h5f[split].keys())
            print(f"\n  /{split}/  ({len(keys)} clips)")
            if keys:
                ex = h5f[split][keys[0]]
                for ds in ("poses", "trans", "betas"):
                    print(f"    {ds:6s}: shape={ex[ds].shape}  dtype={ex[ds].dtype}")
                print(f"    attrs: { dict(ex.attrs) }")


  /train/  (1424 clips)
    poses : shape=(261, 52, 3)  dtype=float32
    trans : shape=(261, 3)  dtype=float32
    betas : shape=(16,)  dtype=float32
    attrs: {'action': 'checking_watch', 'age': np.int64(27), 'framerate': np.int64(120), 'gender': 'male', 'height': np.int64(178), 'mass': np.int64(90), 'n_frames': np.int64(261), 'split': 'train', 'subject': 'Subject_11'}

  /val/  (190 clips)
    poses : shape=(521, 52, 3)  dtype=float32
    trans : shape=(521, 3)  dtype=float32
    betas : shape=(16,)  dtype=float32
    attrs: {'action': 'checking_watch', 'age': np.int64(26), 'framerate': np.int64(120), 'gender': 'male', 'height': np.int64(178), 'mass': np.int64(77), 'n_frames': np.int64(521), 'split': 'val', 'subject': 'Subject_13'}

  /test/  (187 clips)
    poses : shape=(369, 52, 3)  dtype=float32
    trans : shape=(369, 3)  dtype=float32
    betas : shape=(16,)  dtype=float32
    attrs: {'action': 'checking_watch', 'age': np.int64(26), 'framerate': np.int64(120), 'gender': 'fem

In [26]:
h5file = "Gmovi.h5"
fs = 120
with h5py.File(h5file,"r") as h5f: 
     for split in ("train", "val", "test"):
          p = True
          keys = list(h5f[split].keys())
          n_clips = len(keys)
          print(f"\n  /{split}/  ({len(keys)} clips)")
          seconds = 0
          print(type(keys))
          print(type(h5f[split]))
          for key in keys:
               ex = h5f[split][key]
               seconds += ex["trans"].shape[0]/fs
               if p:
                    print(type(key))
                    print(type(ex))
                    print(type(ex["trans"]))
                    p = False

          print(f'{seconds} seconds of data')
          print(f'{seconds/60} minutes of data')
          print(f'{seconds/3600} hours of data')
          print(f'{seconds/n_clips} avg seconds per clip')


  /train/  (1424 clips)
<class 'list'>
<class 'h5py._hl.group.Group'>
<class 'str'>
<class 'h5py._hl.group.Group'>
<class 'h5py._hl.dataset.Dataset'>
8058.46666666667 seconds of data
134.30777777777783 minutes of data
2.2384629629629638 hours of data
5.659035580524347 avg seconds per clip

  /val/  (190 clips)
<class 'list'>
<class 'h5py._hl.group.Group'>
<class 'str'>
<class 'h5py._hl.group.Group'>
<class 'h5py._hl.dataset.Dataset'>
1042.9833333333331 seconds of data
17.38305555555555 minutes of data
0.28971759259259255 hours of data
5.489385964912279 avg seconds per clip

  /test/  (187 clips)
<class 'list'>
<class 'h5py._hl.group.Group'>
<class 'str'>
<class 'h5py._hl.group.Group'>
<class 'h5py._hl.dataset.Dataset'>
1037.9250000000004 seconds of data
17.298750000000005 minutes of data
0.28831250000000014 hours of data
5.550401069518719 avg seconds per clip


In [ ]:
h5file = "Gmovi.h5"
fs = 120
with h5py.File(h5file,"r") as h5f: 
    split = "test"
    p = True
    keys = list(h5f[split].keys())
    n_clips = len(keys)
    # print(type(keys[0]))
    # print(keys[0])
    print(h5f[split])
    # for key in keys:-- here
    #     print(key)
    # print(f"\n  /{split}/  ({len(keys)} clips)")
    # seconds = 0
    # print(type(keys))
    # print(type(h5f[split]))
    # for key in keys:
    #     ex = h5f[split][key]
    #     seconds += ex["trans"].shape[0]/fs
    #     if p:
    #         print(type(key))
    #         print(type(ex))
    #         print(type(ex["trans"]))
    #         p = False

    # print(f'{seconds} seconds of data')
    # print(f'{seconds/60} minutes of data')
    # print(f'{seconds/3600} hours of data')
    # print(f'{seconds/n_clips} avg seconds per clip')

<HDF5 group "/test" (187 members)>
Subject_12__checking_watch
Subject_12__crawling
Subject_12__cross_legged_sitting
Subject_12__crossarms
Subject_12__hand_clapping
Subject_12__hand_waving
Subject_12__jumping_jack
Subject_12__kicking
Subject_12__phone_talking
Subject_12__pointing
Subject_12__running_in_spot
Subject_12__scratching_head
Subject_12__sideways
Subject_12__sitting_down
Subject_12__stretching
Subject_12__taking_photo
Subject_12__throw_catch
Subject_12__vertical_jumping
Subject_12__walking
Subject_12__yoga_rm
Subject_21__checking_watch
Subject_21__crawling
Subject_21__cross_arms
Subject_21__cross_legged_sitting
Subject_21__hand_clapping
Subject_21__hand_waving
Subject_21__jogging
Subject_21__jumping_jacks
Subject_21__kicking
Subject_21__phone_talking
Subject_21__pointing
Subject_21__rowing_rm
Subject_21__running_in_spot
Subject_21__scratching_head
Subject_21__sideways
Subject_21__sitting_down
Subject_21__stretching
Subject_21__taking_photo
Subject_21__throw_catch
Subject_21__ve